# BibLM

An offline, LLM using RAG tag fine tune answers to three translations of the Bible into English.

Shout out to Jeremy K and his [article](https://medium.com/aimonks/exploring-offline-rag-with-langchain-zephyr-7b-beta-and-decilm-7b-c0626e09ee1f) on Medium, and the accompanying [notebook](https://github.com/jeremy-k3/notebooks/blob/main/RAG_Langchain_Zephyr_DeciLM.ipynb), from which this project is largely derived.

---

## Imports

In [1]:
# --- LangChain Core & Utils ---
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.prompts import PromptTemplate
from operator import itemgetter

# --- Text Splitting & Vector Stores ---
from langchain_community import document_loaders as dl
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community import vectorstores as vs

# --- HuggingFace Integrations (Recommended Package) ---
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import HuggingFaceEmbeddings

# --- Standard ML Libraries ---
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

In [2]:
import os
from pathlib import Path
import re

---

## Process and Store RAG Inputs

### Load Translations

In [3]:
doc_dir = Path("/home/benjaminvandewater/Data/BibLM")

In [4]:
def process_translation(doc_path, chunk_size=500, chunk_overlap=20):
    loader = dl.TextLoader(doc_path)
    document = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    document_split = text_splitter.split_documents(document)
    return document_split

In [5]:
erv = process_translation(doc_dir / "ERV.txt")
kjv = process_translation(doc_dir / "KJV.txt")
slt = process_translation(doc_dir / "SLT.txt")

### Load Embedding Model

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model.save("sentence-transformers")
del model
torch.cuda.empty_cache()

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -2] Name or service not known)"))'), '(Request ID: 16068865-4190-4d02-b163-0e3be1c65cce)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -2] Name or service not known)"))'), '(Request ID: 2f012c65-e646-4230-8fc7-40c8e0759fb7)')' thrown while requesting HEAD https://huggingface.co/sentence-transfo

In [8]:
def load_embedding_model():
    model_kwargs = {
        "device": "cuda:0",
        "model_kwargs": {"dtype": torch.float16}}
    encode_kwargs = {"normalize_embeddings": False}
    embedding_model_instance = HuggingFaceEmbeddings(
        model_name="sentence-transformers",
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embedding_model_instance

embedding_model_instance = load_embedding_model()

### Store Embeddings in Vector Database

In [9]:
def create_db(document_split, embedding_model_instance):
    model_vectorstore = vs.FAISS
    db = None
    try:
        content = []
        metadata = []
        for d in document_split:
            content.append(d.page_content)
            metadata.append({"source": d.metadata})
        db = model_vectorstore.from_texts(content, embedding_model_instance, metadata)
    except Exception as error:
        print(error)
    return db

db = create_db(kjv, embedding_model_instance)
db.save_local("db.index")

---

## Load the Base LLM

In [10]:
from transformers import BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

def load_zephyr_on_3060():
    model_id = "HuggingFaceH4/zephyr-7b-beta"
    
    # This config is the magic that makes it fit on 6GB
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto", # Automatically balances between GPU/CPU
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Wrap it for LangChain
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)
    return HuggingFacePipeline(pipeline=pipe)

In [11]:
# tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
# model = AutoModelForCausalLM.from_pretrained("zephyr-7b-beta-model", low_cpu_mem_usage=True, dtype=torch.float16)
# pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, device="cuda:0", max_new_tokens=1000)

---

## 3 - Assemble Pipeline

In [12]:
llm = load_zephyr_on_3060()

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


---

## 4 - Retrieve Relevant Text and Query

In [13]:
def query(q):
    retriever = db.as_retriever(search_type="mmr",
                                search_kwargs={"k": 10, "score_threshold": .01})
    retrieved_docs = retriever.invoke(q)

    template = """
    Use the following piece of context to answer the question at the end.
    If you don't know the answer, just say that you don't know; don't try
    to make up an answer.
    {context}
    
    Question: {question}
    
    Helpful Answer:
    """
    rag_prompt_custom = PromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    #First chain to query the LLM
    rag_chain_from_docs = (
        {
            "context": lambda input: format_docs(input["documents"]),
            "question": itemgetter("question"),
        }
        | rag_prompt_custom
        | llm
        | StrOutputParser()
    )
    
    #Second chain to postprocess the answer
    rag_chain_with_source = RunnableParallel(
        {"documents": retriever, "question": RunnablePassthrough()}
    ) | {
        "documents": lambda input: [doc.metadata for doc in input["documents"]],
        "answer": rag_chain_from_docs,
    }

    resp = rag_chain_with_source.invoke(q)
    if len(resp['documents'])==0:
      print('No documents found')
    else:
      stripped_resp = re.sub(r"\n+$", " ", resp['answer'])
      print(stripped_resp)
      print('Sources', resp['documents'])

In [ ]:
query("What does the Bible say is the most important commandment?")